# Sentiment Classification des Leaders Politiques — Archelec
Fine-tuning CamemBERT sur les **200 mentions présidentielles annotées** (2 classes : positif / négatif).

**Pipeline :**
1. Chargement et exploration des 200 lignes annotées
2. Fine-tuning `camembert-base` → `CamembertForSequenceClassification` (2 classes, class weights)
3. Évaluation : classification report + matrice de confusion
4. Analyse : sentiment par président, par année, par bloc politique

## 0 — Installation des dépendances

In [5]:
!pip install -q transformers torch accelerate scikit-learn openpyxl seaborn

## 1 — Imports & configuration

In [6]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Chemins ───────────────────────────────────────────────────────────────────
NOTEBOOK_DIR   = Path(os.getcwd())
ROOT_DIR       = NOTEBOOK_DIR.parent

OUTPUT_DIR     = ROOT_DIR / "data" / "results" / "output_best_model"
INPUT_FILE     = OUTPUT_DIR / "leaders_mentions.xlsx"
FINAL_FILE     = OUTPUT_DIR / "leaders_sentiment_final.xlsx"
# Sauvegarde dans data/ — évite la dépendance au symlink models/ (cible éphémère)
MODEL_SAVE_DIR = ROOT_DIR / "data" / "results" / "sentiment_model_best"
GRAPHS_DIR     = OUTPUT_DIR / "graphs"

for d in [MODEL_SAVE_DIR, GRAPHS_DIR]:
    os.makedirs(d, exist_ok=True)

LABEL_MAP  = {"négatif": 0, "positif": 1}
ID2LABEL   = {0: "négatif", 1: "positif"}
LABEL_LIST = ["négatif", "positif"]
COLORS     = {"négatif": "#e74c3c", "positif": "#2ecc71"}

print(f"INPUT  : {INPUT_FILE}")
print(f"MODEL  : {MODEL_SAVE_DIR}")
print(f"GRAPHS : {GRAPHS_DIR}")


INPUT  : /home/onyxia/work/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/data/results/output_best_model/leaders_mentions.xlsx
MODEL  : /home/onyxia/work/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/data/results/sentiment_model_best
GRAPHS : /home/onyxia/work/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/data/results/output_best_model/graphs


## 2 — Chargement et exploration des 200 lignes annotées

In [7]:
df_all = pd.read_excel(INPUT_FILE)

# Extraire uniquement les lignes avec un label valide
df = df_all[df_all["sentiment_president"].isin(["positif", "négatif"])].copy()
df["label"] = df["sentiment_president"].map(LABEL_MAP)

print(f"Lignes annotées : {len(df)}")
print()
print("Distribution des labels :")
print(df["sentiment_president"].value_counts().to_string())
print()
print("Par président :")
print(df.groupby(["actual_president", "sentiment_president"]).size().to_string())
df.head(3)


Lignes annotées : 200

Distribution des labels :
sentiment_president
positif    186
négatif     14

Par président :
actual_president  sentiment_president
Georges Pompidou  négatif                 14
                  positif                186


,annee,departement,parti,file_id,entite,actual_president,text,sentiment_president,label
0,1973,Ain,non mentionné,EL065_L_1973_03_001_01_1_PF_05,Georges POMPIDOU,Georges Pompidou,"étiquette, avec la même volonté d'être utile ...",positif,1
1,1973,Ain,Républicain indépendant;Union des républicains...,EL065_L_1973_03_001_02_1_PF_01,Georges POMPIDOU,Georges Pompidou,"rages, accompagné de mon fidèle ami M. Michel ...",positif,1
2,1973,Aisne,Alliance républicaine indépendante et libérale,EL065_L_1973_03_002_01_1_PF_05,Georges Pompidou,Georges Pompidou,tion de l'Aisne\nGeorges LESIMON Commerçant CI...,positif,1


## 3 — Fine-tuning CamemBERT
- `camembert-base` → `CamembertForSequenceClassification` (2 classes)
- Input : colonne `text` (~300 chars de contexte, entité entre `[…]`)
- `max_length=256`, split stratifié 80/10/10
- **Class weights** pour compenser le déséquilibre (186 positif / 14 négatif)
- Meilleur modèle sauvegardé dans `models/sentiment_model_best/`

In [8]:
import torch
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    CamembertTokenizer,
    CamembertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

# ── Split stratifié 80 / 10 / 10 ─────────────────────────────────────────────
X_train, X_temp, y_train, y_temp = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.20,
    stratify=df["label"].tolist(),
    random_state=42,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train : {len(X_train)}  |  Val : {len(X_val)}  |  Test : {len(X_test)}")
print(f"  Train labels : { {ID2LABEL[l]: y_train.count(l) for l in LABEL_MAP.values()} }")


/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train : 160  |  Val : 20  |  Test : 20
  Train labels : {'négatif': 11, 'positif': 149}


In [9]:
MAX_LENGTH = 256
tokenizer  = CamembertTokenizer.from_pretrained("camembert-base")

class SentimentDataset(TorchDataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = SentimentDataset(X_train, y_train)
val_dataset   = SentimentDataset(X_val,   y_val)
test_dataset  = SentimentDataset(X_test,  y_test)
print("Datasets créés.")


Datasets créés.


In [ ]:
# ── Class weights pour le déséquilibre positif/négatif ───────────────────────
raw_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=np.array(y_train),
)
class_weights = torch.tensor(raw_weights, dtype=torch.float)
print(f"Class weights : négatif={raw_weights[0]:.2f}  positif={raw_weights[1]:.2f}")

# ── Modèle ────────────────────────────────────────────────────────────────────
model_ft = CamembertForSequenceClassification.from_pretrained(
    "camembert-base",
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL_MAP,
)

# Trainer custom pour injecter les class weights dans la loss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1_macro": float(f1_score(labels, preds, average="macro", zero_division=0)),
    }

# CUDA driver trop vieux sur cet environnement → CPU uniquement
use_fp16 = False

training_args = TrainingArguments(
    output_dir=str(MODEL_SAVE_DIR),
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",       # evaluation_strategy renommé en eval_strategy (transformers ≥ 4.46)
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    fp16=use_fp16,
    warmup_ratio=0.1,
    no_cuda=True,                # forcer CPU (driver CUDA incompatible)
)

trainer = WeightedTrainer(
    model=model_ft,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Device : CPU (driver CUDA incompatible avec cette version de PyTorch)")
trainer.train()


In [ ]:
trainer.save_model(str(MODEL_SAVE_DIR))
tokenizer.save_pretrained(str(MODEL_SAVE_DIR))
print(f"Modèle sauvegardé : {MODEL_SAVE_DIR}")


## 4 — Évaluation sur le test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred_output   = trainer.predict(test_dataset)
y_pred        = np.argmax(pred_output.predictions, axis=-1)
y_pred_labels = [ID2LABEL[p] for p in y_pred]
y_true_labels = [ID2LABEL[t] for t in y_test]

print("=" * 50)
print("Classification Report — Test Set")
print("=" * 50)
print(classification_report(y_true_labels, y_pred_labels, target_names=LABEL_LIST))


In [ ]:
cm = confusion_matrix(y_true_labels, y_pred_labels, labels=LABEL_LIST)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=LABEL_LIST, yticklabels=LABEL_LIST, ax=ax,
)
ax.set_xlabel("Prédiction")
ax.set_ylabel("Vérité terrain")
ax.set_title("Matrice de confusion — Test Set")
plt.tight_layout()
plt.savefig(GRAPHS_DIR / "confusion_matrix.png", dpi=150)
plt.show()


## 5 — Analyse & visualisations

In [ ]:
# ── Mapping bloc politique ────────────────────────────────────────────────────
GAUCHE_KW = [
    "socialiste", "communiste", "lutte ouvrière", "parti socialiste unifié",
    "radicaux de gauche", "sfio", "gauche", "trotsk", "ligue communiste",
]
DROITE_KW = [
    "rassemblement pour la République", "rpr", "républicain indépendant",
    "union des républicains", "front national", "alliance républicaine",
    "gaulliste", "cnip", "droite",
]
CENTRE_KW = [
    "union pour la démocratie", "udf", "centre démocrate", "mrp", "cds",
    "centre", "réformateur", "libéral", "radical",
]

def get_bloc(parti):
    if not isinstance(parti, str):
        return "non classé"
    p = parti.lower()
    if any(kw in p for kw in GAUCHE_KW):
        return "gauche"
    if any(kw in p for kw in DROITE_KW):
        return "droite"
    if any(kw in p for kw in CENTRE_KW):
        return "centre"
    return "non classé"

df["bloc_politique"] = df["parti"].apply(get_bloc)
print("Bloc politique :")
print(df["bloc_politique"].value_counts().to_string())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Par président ─────────────────────────────────────────────────────────────
pres = (
    df.groupby(["actual_president", "sentiment_president"])
    .size().unstack(fill_value=0)
    .reindex(columns=LABEL_LIST, fill_value=0)
)
pres.plot(kind="bar", ax=axes[0], color=[COLORS[l] for l in LABEL_LIST], edgecolor="white")
axes[0].set_title("Sentiment par président")
axes[0].set_xlabel("")
axes[0].set_ylabel("Mentions")
axes[0].legend(title="Sentiment")
axes[0].tick_params(axis="x", rotation=20)

# ── Par année ─────────────────────────────────────────────────────────────────
annee = (
    df.groupby(["annee", "sentiment_president"])
    .size().unstack(fill_value=0)
    .reindex(columns=LABEL_LIST, fill_value=0)
    .sort_index()
)
annee.plot(kind="bar", stacked=True, ax=axes[1], color=[COLORS[l] for l in LABEL_LIST], edgecolor="white")
axes[1].set_title("Sentiment par année")
axes[1].set_xlabel("Année")
axes[1].set_ylabel("Mentions")
axes[1].legend(title="Sentiment")
axes[1].tick_params(axis="x", rotation=0)

# ── Par bloc politique ────────────────────────────────────────────────────────
bloc = (
    df.groupby(["bloc_politique", "sentiment_president"])
    .size().unstack(fill_value=0)
    .reindex(index=["gauche","centre","droite","non classé"], columns=LABEL_LIST, fill_value=0)
)
bloc.plot(kind="bar", ax=axes[2], color=[COLORS[l] for l in LABEL_LIST], edgecolor="white")
axes[2].set_title("Sentiment par bloc politique")
axes[2].set_xlabel("")
axes[2].set_ylabel("Mentions")
axes[2].legend(title="Sentiment")
axes[2].tick_params(axis="x", rotation=10)

plt.tight_layout()
plt.savefig(GRAPHS_DIR / "sentiment_analyse.png", dpi=150)
plt.show()
print(f"Graphique sauvegardé : {GRAPHS_DIR / 'sentiment_analyse.png'}")


In [ ]:
# ── Sauvegarde finale ─────────────────────────────────────────────────────────
df.to_excel(FINAL_FILE, index=False, engine="openpyxl")
print(f"Fichier sauvegardé : {FINAL_FILE}")
print(f"  Lignes    : {len(df)}")
print(f"  Colonnes  : {df.columns.tolist()}")
